In [0]:
import logging
logging.disable(logging.CRITICAL)

In [0]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import random
import time
import logging
import re
from urllib.parse import quote

# ----------------- Logging Setup -----------------
logging.basicConfig(
    level=logging.DEBUG,   # Change to INFO if you want less verbosity
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger(__name__)
# -------------------------------------------------

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}


class LinkedInScraper:
    def __init__(self, keywords, locations):
        """
        keywords: string
        locations: string or list of strings
        """
        self.keywords = keywords
        if isinstance(locations, str):
            self.locations = [locations]
        else:
            self.locations = locations

        logger.info(f"Initialized scraper for keywords: '{self.keywords}' and locations: {self.locations}")

    def get_job_ids(self, base_url):
        """Extract job IDs from search result page for a given URL"""
        logger.info(f"Fetching job IDs from URL: {base_url}")
        response = requests.get(base_url, headers=HEADERS)
        logger.debug(f"Response status code: {response.status_code}")

        soup = BeautifulSoup(response.text, "html.parser")
        jobs = soup.find_all("li")
        logger.debug(f"Found {len(jobs)} job list items in HTML")

        job_ids = []
        for job in jobs:
            try:
                base_card_div = job.find("div", {"class": "base-card"})
                job_id = base_card_div.get("data-entity-urn").split(":")[3]
                job_ids.append(job_id)
                logger.debug(f"Extracted Job ID: {job_id}")
            except Exception as e:
                logger.warning(f"Failed to extract job id from job element: {e}")
                continue

        logger.info(f"Total job IDs extracted: {len(job_ids)}")
        return job_ids

    def extract_responsibilities_and_skills(self, soup):
        """Extract responsibilities, skills, and raw description"""
        logger.debug("Extracting responsibilities and skills...")
        description_div = soup.select_one("div.show-more-less-html__markup")
        responsibilities, skills, raw_description = [], [], None

        if description_div:
            raw_description = description_div.get_text(" ", strip=True)
            current_section = None

            for elem in description_div.children:
                # Detect section headers
                if elem.name == "p":
                    text = elem.get_text(strip=True)
                    if "responsibilit" in text.lower():
                        current_section = "responsibilities"
                        logger.debug("Found Responsibilities section")
                    elif "skill" in text.lower():
                        current_section = "skills"
                        logger.debug("Found Skills section")

                # Collect bullet points
                elif elem.name == "ul":
                    bullets = [li.get_text(strip=True) for li in elem.find_all("li")]
                    if current_section == "responsibilities":
                        responsibilities.extend(bullets)
                        logger.debug(f"Responsibilities bullets: {bullets}")
                    elif current_section == "skills":
                        skills.extend(bullets)
                        logger.debug(f"Skills bullets: {bullets}")

        # ✅ Fallback: split description into sentences if no bullets found
        if not responsibilities and not skills and raw_description:
            sentences = [s.strip() for s in re.split(r"[.!?]", raw_description) if s.strip()]
            responsibilities = sentences
            logger.debug("No bullets found, used fallback sentence split")

        return responsibilities, skills, raw_description

    def get_job_details(self, job_id, location):
        """Extract job details from job posting page"""
        logger.info(f"Fetching details for Job ID: {job_id} at location: {location}")
        job_url = f"https://www.linkedin.com/jobs-guest/jobs/api/jobPosting/{job_id}"
        response = requests.get(job_url, headers=HEADERS)
        logger.debug(f"Response status code for job {job_id}: {response.status_code}")

        soup = BeautifulSoup(response.text, "html.parser")
        job_post = {}

        def safe_extract(selector, multiple=False):
            try:
                if multiple:
                    values = [s.text.strip() for s in soup.select(selector)]
                    logger.debug(f"Extracted multiple values for {selector}: {values}")
                    return values
                value = soup.select_one(selector).text.strip()
                logger.debug(f"Extracted value for {selector}: {value}")
                return value
            except Exception as e:
                logger.warning(f"Failed to extract {selector}: {e}")
                return None

        # Base job info
        job_post["job_id"] = job_id
        job_post["job_title"] = safe_extract("h2.top-card-layout__title")
        job_post["company_name"] = safe_extract("a.topcard__org-name-link")
        job_post["time_posted"] = safe_extract("span.posted-time-ago__text")
        job_post["num_applicants"] = safe_extract("span.num-applicants__caption")
        job_post["location"] = location

        # Criteria
        criteria = safe_extract("span.description__job-criteria-text", multiple=True)
        if criteria and len(criteria) >= 4:
            job_post["seniority_level"] = criteria[0]
            job_post["employment_type"] = criteria[1]
            job_post["job_function"] = criteria[2]
            job_post["industries"] = criteria[3]
        else:
            job_post["seniority_level"] = None
            job_post["employment_type"] = None
            job_post["job_function"] = None
            job_post["industries"] = None

        # ✅ Extract responsibilities & skills
        responsibilities, skills, raw_desc = self.extract_responsibilities_and_skills(soup)
        job_post["responsibilities"] = responsibilities
        job_post["skills"] = skills
        job_post["raw_description"] = raw_desc

        logger.info(f"Job details extracted for {job_id}")
        return job_post

    def scrape_jobs(self):
        """Main method to scrape jobs for all locations"""
        logger.info("Starting job scraping process for all locations...")
        all_jobs = []

        for loc in self.locations:
            # URL encode keywords and location
            loc_encoded = quote(loc)
            keywords_encoded = quote(self.keywords)
            base_url = f"https://www.linkedin.com/jobs/search?keywords={keywords_encoded}&location={loc_encoded}&position=1&pageNum=0"
            logger.info(f"Scraping jobs for location: {loc} | URL: {base_url}")

            job_ids = self.get_job_ids(base_url)

            for job_id in job_ids:
                details = self.get_job_details(job_id, loc)
                all_jobs.append(details)
                logger.info(f"Appended job details for {job_id} at location {loc}")
                time.sleep(random.uniform(1, 3))  # polite scraping

        logger.info(f"Scraping complete. Total jobs scraped across all locations: {len(all_jobs)}")
        return all_jobs


In [0]:
import pandas as pd
from scraper import LinkedInScraper

if __name__ == "__main__":
    # List of locations to scrape
    locations = ["India", "United States", "Germany"]

    # Create scraper instance with multiple keywords and locations
    scraper = LinkedInScraper(
        keywords="Data Engineer OR Data Analyst OR Machine Learning Engineer",
        locations=locations
    )

    # Scrape jobs
    jobs = scraper.scrape_jobs()

    # Convert to DataFrame
    df = pd.DataFrame(jobs)

    # Display first 5 rows
    print(df.head())

In [0]:
job_ids = scraper.get_job_ids()
print("Extracted Job IDs:", job_ids)
print("Total Jobs Found:", len(job_ids))

if not job_ids:
    print(scraper.base_url)
    response = requests.get(scraper.base_url, headers=HEADERS)
    print(response.text[:1000])

In [0]:
if job_ids:
    sample_job_id = job_ids[20]
    print("Testing with Job ID:", sample_job_id)

    job_details = scraper.get_job_details(sample_job_id)
    print("Job Details Extracted:")
    print(job_details)

In [0]:
# Keywords
keywords = "Data Engineer"

# List of locations
locations = ["India", "United States"]

for loc in locations:
    # Encode the location for URL (replace spaces with %20)
    loc_encoded = loc.replace(" ", "%20")
    base_url = f"https://www.linkedin.com/jobs/search?keywords={keywords.replace(' ', '%20')}&location={loc_encoded}&position=1&pageNum=0"
    print(f"Base URL for {loc}: {base_url}")


In [0]:
keywords = "Data Engineer"
locations = ["India", "United States"]

scraper = LinkedIn(keywords, locations)
all_jobs = scraper.scrape_jobs()
print(all_jobs[:5])

In [0]:
from bs4 import BeautifulSoup

html = """
<div class="core-section-container content break-words">
  <div class="description text description">
    <section class="show-more-less-html" data-max-lines="5">
      <div class="show-more-less-html_markup show-more-less-html relative overflow-hidden">
        location :
        "Hyderabad, India"
      </div>
    </section>
  </div>
</div>
"""

soup = BeautifulSoup(html, "html.parser")

# Select the inner div containing the location
location_div = soup.select_one("div.show-more-less-html_markup")
if location_div:
    text = location_div.get_text(strip=True)
    # Extract the location part after 'location :'
    import re
    match = re.search(r'location\s*:\s*"?(.*?)"?$', text, re.IGNORECASE)
    if match:
        city = match.group(1).split(",")[0]
        print("city:", city)

In [0]:
import logging

# Basic configuration
logging.basicConfig(
    level=logging.DEBUG,   # Minimum log level
    format="%(asctime)s - %(levelname)s - %(message)s",
    filename="app.log",    # Write logs to file
    filemode="a"           # Append mode
)

# Example logs
logging.debug("Debugging info")
logging.info("Service started")
logging.warning("Low disk space")
logging.error("Error connecting to database")
logging.critical("System crash")

In [0]:
import logging

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

print(logging.debug("Debugging info"))
logging.info("Service started")
logging.warning("Low disk space")
logging.error("Error connecting to database")
logging.critical("System crash")